### **Enviornment**

In [3]:
# Setting up the local imports and env variables
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ['LANGCHAIN_TRACING_V2'] = os.getenv("LANGCHAIN_TRACING_V2")
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv("LANGCHAIN_ENDPOINT")
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

# **1. Multi Query** 

In [4]:
# Indexing
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI , OpenAIEmbeddings
from langsmith import Client
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import OllamaEmbeddings , ChatOllama


loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

# Split
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size = 300,chunk_overlap=50)

# Make split
splits = text_splitter.split_documents(blog_docs)

# Embed
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vectorstore = Chroma.from_documents(documents=splits , embedding=embeddings)
retriver = vectorstore.as_retriever()

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_2204\1623085948.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

llm = ChatOllama(model="llama3.2:3b", temperature=0.2)

generate_queries = (
    prompt_perspectives | 
    llm | StrOutputParser() | (lambda x : x.split("\n"))
)

In [6]:
from langchain_core.load import dumps, loads 
def get_unique_union(documents : list[list]):
    """Unique union of required doc"""
    # Flatten list of lists and convert each document into string 
    flattended_doc = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattended_doc))
    # Return
    return [loads(doc) for doc in unique_docs]

In [7]:
# Retrieve
question = "What is task decomposition for LLM agents?"
retrieval_chain = generate_queries | retriver.map() | get_unique_union
docs = retrieval_chain.invoke({"question":question})
len(docs)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_2204\305651393.py:9: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]


12

In [8]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

# RAG
template = """Answer the following question based on this context:
{context}
Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template=template)

final_rag_chain = (
    {"context" : retrieval_chain , 
     "question" : itemgetter("question")} | 
     prompt | llm | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

'Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down complex tasks into smaller, manageable subgoals. This allows the agent to efficiently handle complex tasks and enables efficient handling of large tasks by decomposing them into multiple simpler steps. The goal is to transform big tasks into multiple manageable tasks, shedding light on an interpretation of the model\'s thinking process.\n\nThere are three ways task decomposition can be done:\n\n1. By LLM with simple prompting like "Steps for XYZ. 1.", "What are the subgoals for achieving XYZ?"\n2. Using task-specific instructions; e.g., "Write a story outline." for writing a novel\n3. With human inputs'

# **2. RAG Fusion** 

In [9]:
template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

llm = ChatOllama(model="llama3.2:3b", temperature=0.2)

generate_queries = (
    prompt_rag_fusion | 
    llm | StrOutputParser() | (lambda x : x.split("\n"))
)

In [10]:
from langchain_core.load import dumps,loads 

def reciprocal_rank_fusion(results : list[list] , k = 60):
    """Reciprocal Rank fusion that takes multiple lists of ranked documents and 
       a optimal parameter k used in the RRF formula.
    """
    # Initialize a dictionary to hold fused score for each unique document
    fused_score = {}

    # iterate through each list of ranked documents
    for rank,doc in enumerate(docs):
        # Convert the document to a string format to use as a key
        doc_str = dumps(doc)
        # If the document is not yet in the fused_score dict then add it with initial score of 0
        if doc_str not in fused_score:
            fused_score[doc_str] = 0
            previous_score = fused_score[doc_str]
            fused_score[doc_str] += 1 / (rank + k)

    # Sort the documents basedo n their fused scores in descending order to get the final result
    reranked_results = [
        (loads(doc) , score)
        for doc,score in sorted(fused_score.items() , key = lambda x : x[1] , reverse=True)
    ]

    return reranked_results

retrieval_chain_rag_fusion = generate_queries | retriver.map() | reciprocal_rank_fusion
docs = retrieval_chain_rag_fusion.invoke({"question" : question})
len(docs)

12

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# RAG
template = """Answer the following question based on this context:
{context}
Question : {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context" : retrieval_chain_rag_fusion,
     "question" : itemgetter("question")} | 
     prompt | llm | StrOutputParser()
)

final_rag_chain.invoke({"question" : question})

'Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down large tasks into smaller, manageable subgoals. This allows the agent to efficiently handle complex tasks and improve the quality of its final results.\n\nIn the context of LLM-powered autonomous agents, task decomposition typically involves:\n\n1. Subgoal identification: The agent identifies specific subgoals that need to be achieved in order to complete the larger task.\n2. Decomposition: The agent breaks down each subgoal into smaller, more manageable tasks or steps.\n3. Planning: The agent plans how to achieve each of these smaller tasks and decomposes them further if necessary.\n\nThe goal of task decomposition is to enable the LLM agent to efficiently handle complex tasks by:\n\n* Reducing the complexity of the task\n* Improving the quality of the final results\n* Allowing for more efficient handling of large tasks\n\nTask decomposition can be done using various techniques, such as:\n\

# **3. Decomposition** 

In [21]:
template = """You are a helpful assistant that generate multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-probelms / sub questions that can be answers in isolation. \n
output only be querys nothing else.Generate multiple search queries related to : {question}\n
Output (3 queries):"""
prompt_decomposition = ChatPromptTemplate.from_template(template)

In [22]:
# LLM
llm = ChatOllama(model="llama3.2:3b", temperature=0.2)

# Chain
generate_query_decomposition = (
    prompt_decomposition
    | llm
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

# Run the chain
question = "What are the main components of a LLM-powered autonomous agent system?"
questions = generate_query_decomposition.invoke({"question" : question})

In [25]:
questions

['1. What is the role of Natural Language Processing (NLP) in a LLM-powered autonomous agent system?',
 '2. How do Large Language Models (LLMs) contribute to decision-making and reasoning in an autonomous agent system?',
 "3. What are the key components of a LLM-powered autonomous agent system's perception and sensor integration?"]

In [26]:
# Prompt
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question: 

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

In [27]:
from operator import itemgetter

def format_qa_pair(question , answer):
    """ Format QA Pair"""
    formatted_string = ""
    formatted_string += f"Question : {question}\nAnswer : {answer}\n\n"
    return formatted_string.strip()

# LLM
llm = ChatOllama(model="llama3.2:3b", temperature=0.2)

q_a_pairs = ""
for q in questions:
    rag_chain = (
        {"context" : itemgetter("question") | retriver ,
        "question" : itemgetter("question"),
        "q_a_pairs" : itemgetter("q_a_pairs")}
        | decomposition_prompt | llm | StrOutputParser()
    )

    answer = rag_chain.invoke({"question" : q , "q_a_pairs" : q_a_pairs})
    q_a_pair = format_qa_pair(q , answer)
    q_a_pairs = q_a_pairs + "\n\n" + q_a_pair

In [28]:
answer

'Based on the provided context, the key components of a LLM-powered autonomous agent system\'s perception and sensor integration can be inferred as follows:\n\n1. **Short-term memory**: The agent uses short-term memory to learn from in-context information, which is utilized by the model to learn.\n2. **Long-term memory**: The agent has long-term memory capabilities, which provide the capability to retain and recall (infinite) information over extended periods, often by leveraging an external vector store and fast retrieval.\n\nThese components are mentioned in the context of the LLM-powered autonomous agent system\'s architecture, specifically under the "Memory" section. While they do not directly relate to perception and sensor integration, they can be seen as part of the overall system that enables the agent to interact with its environment.\n\nHowever, it is worth noting that the provided background question + answer pairs do not explicitly mention perception and sensor integration 

In [31]:
prompt_rag = ChatPromptTemplate.from_template("""
You are a knowledgeable AI assistant.

Answer the following sub-question using ONLY the provided context.

If the context does not contain enough information, reply with:
"I don't have enough information in the provided context."

Context:
--------------------
{context}
--------------------

Sub-question:
{question}

Provide a clear and concise answer.
""")

def retrive_and_rag(question , prompt_rag,sub_question_generator_chain):
    """RAG on each sub question"""

    sub_questions = sub_question_generator_chain.invoke({"question" : question})

    # Init alist to hold the rag chain result 
    rag_results = []

    for sub_question in sub_questions:
        
        # Retrive Documents for each sub question
        retrived_docs = retriver.invoke(sub_question)

        # Use retrived documents and sub-question in RAG chain
        answer = (prompt_rag | llm | StrOutputParser()).invoke({"context" : retrived_docs , "question" : sub_question})

        rag_results.append(answer)
    
    return rag_results , sub_questions 

# Wrap the retrival and RAG process in Runnable Lambda for integration into chain
answers , questions = retrive_and_rag(question , prompt_rag , generate_query_decomposition)


In [34]:
def format_qa_pairs(questions , answers):
    """Format Q and A pairs"""

    formatted_string = ""
    for i, (question,answer) in enumerate(zip(questions , answers) , start = 1):
        formatted_string += f"Question {i} : {question}\nAnswer {i} : {answer}\n\n"
    return formatted_string.strip()

context = format_qa_pairs(questions , answers)

# Prompt 
template = """Here is a set of Q+A pairs:
{context}

Use these to synthesize an answer to the question: {question}

Formatting rules:
- Use clear Markdown formatting (headings, bullet points)
- Add emojis to improve readability (use sparingly, 1–2 per section max)
- Do NOT overuse emojis
- Keep the answer structured and easy to read
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt | llm | StrOutputParser()
)

print(final_rag_chain.invoke({"context" : context , "question" : question}))

**Main Components of a LLM-Powered Autonomous Agent System**

A Large Language Model (LLM)-powered autonomous agent system is a complex entity that combines multiple components to achieve its goals. Based on the provided context, we can infer the following main components:

### 1. **Planning and Decision-Making** 🤔
The LLM acts as the brain of the agent, making it a powerful general problem solver. It breaks down large tasks into smaller subgoals through planning and reflects on past actions to learn from mistakes and refine them for future steps.

### 2. **Memory and Tool Use**
The system uses memory to store information and tools to interact with external components. This allows the agent to access and process relevant data, making informed decisions.

### 3. **Modular Reasoning, Knowledge, and Language (MRKL) Architecture**
The MRKL architecture is a key component of the LLM-powered autonomous agent system. It enables the agent to reason, learn, and communicate effectively with its 

# **4. Step Back**

In [ ]:
# Few Shot Examples
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# Setting up examples for few shot
examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel’s was born in what country?",
        "output": "what is Jan Sindel’s personal history?",
    },
]

# Setting up the example prompt 
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human","{input}"),
        ("ai" , "{output}")
    ]
)

# Setting up few shot prompt itself using the above two items 
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt , 
    examples = examples
)

# Final prompt we will send to the LLM
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        # Few shot examples
        few_shot_prompt,
        # New question
        ("user", "{question}"),
    ]
)

In [37]:
generate_step_bak_query = prompt | llm | StrOutputParser()
question = "What is task decomposition for LLM agents?"
print(generate_step_bak_query.invoke({"question": question}))

how do large language models process and understand complex tasks? 

(or, more simply: 
what are the basic steps involved in a large language model's processing?)


In [38]:
from langchain_core.runnables import RunnableLambda
# Response prompt 
response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""

response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

chain = (
    {
        # Retrive Context using the normal question
        "normal_context" : RunnableLambda(lambda x : x['question']) | retriver,
        # Retrive context using the stepback question
        "step_back_context" : generate_step_bak_query | retriver,
        # Pass on the question
        "question" : lambda x : x['question']
    } 
    | response_prompt | llm | StrOutputParser()
)

chain.invoke({"question" : question})

'Task decomposition for LLM (Large Language Model) agents is the process of breaking down complex tasks into smaller, manageable subgoals. This allows the agent to efficiently handle large and intricate tasks by identifying individual steps or components that need to be completed.\n\nThere are three main methods for task decomposition in LLM agents:\n\n1. **Simple prompting**: The LLM is instructed with simple prompts such as "Steps for XYZ. 1.", "What are the subgoals for achieving XYZ?", which enables the model to generate a list of smaller tasks.\n2. **Task-specific instructions**: The agent receives task-specific instructions, such as "Write a story outline." for writing a novel, which helps the LLM to focus on specific goals and objectives.\n3. **Human inputs**: Human input is used to provide additional guidance or context to the LLM, helping it to better understand the task and break it down into smaller subgoals.\n\nTask decomposition is an essential component of LLM-powered aut

# **5. HyDE**

In [40]:
# HyDE document genration
template = """Please write a scientific paper passage to answer the question
Question: {question}
Passage:"""
prompt_hyde = ChatPromptTemplate.from_template(template)

generate_docs_for_retrival = prompt_hyde | llm | StrOutputParser()

question = "What is task decomposition for LLM agents?"
generate_docs_for_retrival.invoke({"question":question})

"Here's a potential passage:\n\nTask Decomposition in Large Language Model (LLM) Agents: A Framework for Efficient and Effective Task Execution\n\nIn recent years, Large Language Models (LLMs) have emerged as a dominant paradigm for natural language processing tasks. However, the complexity of these models and their vast capacity for generating human-like text can lead to challenges in task execution. One such challenge is the need to decompose complex tasks into manageable sub-tasks that can be efficiently executed by LLM agents.\n\nTask decomposition refers to the process of breaking down a complex task into smaller, more manageable sub-tasks that can be solved independently by an LLM agent. This approach enables the model to focus on one sub-task at a time, reducing the complexity and computational requirements associated with large-scale task execution. By decomposing tasks in this manner, LLM agents can leverage their vast capacity for generating text to solve individual sub-tasks

In [41]:
# Retrieve
retrieval_chain = generate_docs_for_retrival | retriver 
retireved_docs = retrieval_chain.invoke({"question":question})
retireved_docs

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Or\n@article{weng2023agent,\n  title   = "LLM-powered Autonomous Agents",\n  author  = "Weng, Lilian",\n  journal = "lilianweng.github.io",\n  year    = "2023",\n  month   = "Jun",\n  url     = "https://lilianweng.github.io/posts/2023-06-23-agent/"\n}\nReferences#\n[1] Wei et al. “Chain of thought prompting elicits reasoning in large language models.” NeurIPS 2022\n[2] Yao et al. “Tree of Thoughts: Dliberate Problem Solving with Large Language Models.” arXiv preprint arXiv:2305.10601 (2023).\n[3] Liu et al. “Chain of Hindsight Aligns Language Models with Feedback\n“ arXiv preprint arXiv:2302.02676 (2023).\n[4] Liu et al. “LLM+P: Empowering Large Language Models with Optimal Planning Proficiency” arXiv preprint arXiv:2304.11477 (2023).'),
 Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Component One: Planning#\nA complicated task usual

In [42]:
# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context":retireved_docs,"question":question})

'According to the text, task decomposition for LLM agents involves breaking down large tasks into smaller, manageable subgoals. This can be done in three ways:\n\n1. Using simple prompting techniques, such as "Steps for XYZ. 1." or "What are the subgoals for achieving XYZ?"\n2. Using task-specific instructions, such as "Write a story outline" for writing a novel\n3. With human inputs.\n\nTask decomposition is used to enable efficient handling of complex tasks and is an important component of LLM-powered autonomous agent systems.'